# Agent 품질 평가 실습 — Accuracy · Faithfulness · Robustness

앞서 만든 SQL 에이전트를 **평가 대상**으로 사용합니다.  
Golden Test Set을 만들고, 정량 지표와 LM-as-a-Judge로 품질을 측정합니다.

| 축 | 정의 | 이 실습에서 측정하는 것 |
|---|---|---|
| **Accuracy** | 정답과 얼마나 일치하는가 | SQL 에이전트 응답 vs. 기대 답변 |
| **Faithfulness** | 출처에 근거한 응답인가 | 응답이 DB 쿼리 결과에만 근거하는가 |
| **Robustness** | 입력 변형에 일관된가 | 같은 질문을 다르게 물어도 같은 답이 나오는가 |

### 학습 목표

- Golden Test Set의 구조를 이해하고 직접 작성한다
- 정량 평가 지표(EM, F1)를 구현하고 적용한다
- LM-as-a-Judge로 서술형 응답을 자동 평가한다
- Pairwise 비교로 위치 편향을 제거한다
- 복합 지표 대시보드로 배포 기준을 설정한다

### 전체 흐름

이 노트북은 아래 순서로 진행됩니다. 각 단계마다 직접 코드를 수정하는 실습이 있습니다.

```
1. 환경 설정         SQL Agent + Judge LLM 준비
2. Golden Test Set   Chinook DB에서 정답 확인 → 테스트 케이스 작성
3. 정량 평가         Keyword Match, Token F1로 자동 채점         ← 실습 1
4. LM-as-a-Judge     LLM이 서술형 응답을 0.0~1.0으로 채점
5. Faithfulness      "근거 없이 지어낸 내용은 없는가?"           ← 실습 2
6. Robustness        같은 질문의 변형에 일관된 답을 내는가        ← 실습 3
7. Pairwise 비교     위치 편향 제거 + 프롬프트 A/B 테스트        ← 실습 4
8. 복합 대시보드     배포 가능 여부를 종합 판단                   ← 실습 5
```

---
## 1. 환경 설정

두 가지 LLM을 준비합니다:

1. **평가 대상 (SQL Agent)** — 이전 실습에서 만든 SQL 에이전트. 자연어 질문을 받아 Chinook DB를 조회하고 답변합니다.
2. **평가자 (Judge LLM)** — OpenAI API를 직접 호출하여 에이전트 응답의 품질을 채점합니다.

> 왜 둘을 분리하는가? 같은 모델이 생성하고 평가하면 "자기 선호 편향"이 생깁니다.  
> 생성 모델과 평가 모델을 다르게 설정하는 것이 권장됩니다.

In [6]:
from dotenv import load_dotenv
load_dotenv(override=True)

import os, json
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_agent

# ── 평가 대상: SQL 에이전트 구성 ──
# Chinook DB: 음악 스토어 샘플 DB (Customer, Invoice, Track, Genre 등)
llm = ChatOpenAI(model="gpt-5.4-mini")
db = SQLDatabase.from_uri("sqlite:///Chinook.db")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()  # sql_db_query, sql_db_schema, sql_db_list_tables, sql_db_query_checker

sql_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "당신은 SQL 에이전트입니다. 단계:\n"
        "1. sql_db_list_tables\n2. sql_db_schema\n"
        "3. 쿼리 작성 + sql_db_query_checker\n"
        "4. sql_db_query\n5. 결과를 한국어로 해석하세요.\n"
        f"규칙: LIMIT 10 사용. DML 금지. Dialect: {db.dialect}"
    ),
)

# ── 평가자 (Judge): OpenAI API 직접 호출 ──
# LM-as-a-Judge에서 사용할 클라이언트
client = OpenAI()

print(f"SQL Agent 준비 완료 (DB: {db.dialect}, 테이블: {db.get_usable_table_names()[:5]}...)")
print(f"Judge 모델: gpt-5.4-mini")

SQL Agent 준비 완료 (DB: sqlite, 테이블: ['Album', 'Artist', 'Customer', 'Employee', 'Genre']...)
Judge 모델: gpt-5.4-mini


In [7]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

Langfuse tracing ON — 


### 에이전트 호출 헬퍼

평가에서는 에이전트를 수십 번 반복 호출합니다.  
매번 `invoke → messages[-1].content` 패턴을 쓰는 대신, 질문→응답 텍스트 변환 함수를 만들어 둡니다.

In [15]:
def ask_agent(question: str) -> str:
    """SQL 에이전트에 질문하고 최종 응답 텍스트만 반환"""
    result = sql_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},config=lf_config
    )
    # messages 리스트의 마지막 = 에이전트의 최종 응답
    return result["messages"][-1].content

# 동작 확인 — 아래와 같은 형태의 자연어 답변이 출력되어야 합니다
print(ask_agent("고객이 가장 많은 나라 3개는?"))

고객 수가 가장 많은 나라는 다음 3곳입니다.

1. **USA**: 13명  
2. **Canada**: 8명  
3. **Brazil**: 5명  

참고로 **France**도 5명으로 Brazil과 동률입니다.


---
## 2. Golden Test Set 만들기

Golden Test Set은 **"이건 반드시 맞혀야 한다"** 는 핵심 테스트 모음입니다.  
Chinook DB의 실제 데이터를 기반으로 정답을 미리 확인하고 테스트 케이스를 만듭니다.

### 왜 Golden Test Set인가?

에이전트를 "그냥 몇 번 써보고 괜찮다"는 식으로 평가하면, 배포 후 예상치 못한 곳에서 실패합니다.  
Golden Test Set은 **재현 가능한 기준선**을 제공합니다:
- 코드 변경 전후로 동일한 테스트를 돌려 **회귀(regression)**를 탐지
- 팀 전체가 합의한 **통과 기준**을 숫자로 설정
- CI/CD 파이프라인에 통합하여 **자동 게이트** 역할

### 테스트 케이스 구조

```python
{
    "id": "acc_001",          # 고유 ID (카테고리_번호)
    "category": "accuracy",   # accuracy | faithfulness | robustness
    "question": "...",        # 에이전트에 보낼 질문
    "expected": "...",        # 기대하는 핵심 내용 (LM Judge가 비교에 사용)
    "expected_keywords": [],   # 반드시 포함해야 할 키워드 (정량 평가에 사용)
    "tolerance": 0.8          # 합격 기준 점수 (0.0 ~ 1.0)
}
```

### 작성 순서

```
1. DB에서 직접 SQL을 실행해 정답을 확인한다
2. 정답의 핵심 키워드를 뽑는다 (예: "USA", "13")
3. 에이전트에 보낼 자연어 질문을 작성한다
4. tolerance(합격 기준)를 설정한다
```

### 2.1 정답 데이터 확인

테스트 케이스를 만들기 전에, DB에서 직접 SQL을 실행하여 **정답을 먼저 확인**합니다.  
이 값들이 Golden Test Set의 `expected`와 `expected_keywords`의 근거가 됩니다.

> 중요: 정답을 모르고 테스트를 만들면 의미가 없습니다.  
> "이 질문의 정답은 X이다. 에이전트가 X를 대답하는가?" — 이것이 평가의 기본입니다.

In [29]:
# 정답 확인용 — DB에 직접 SQL 실행하여 ground truth 확보

# 테스트 1: 나라별 고객 수 (acc_001, rob_001, rob_002에서 사용)
print("=== 나라별 고객 수 TOP 5 ===")
print(db.run("SELECT Country, COUNT(*) as cnt FROM Customer GROUP BY Country ORDER BY cnt DESC LIMIT 5"))
# 테스트 2: 장르별 트랙 수 (acc_002에서 사용)
print("\n=== 장르별 트랙 수 TOP 3 ===")
print(db.run("SELECT g.Name, COUNT(t.TrackId) as cnt FROM Genre g JOIN Track t ON g.GenreId = t.GenreId GROUP BY g.Name ORDER BY cnt DESC LIMIT 3"))
# 테스트 3: 가장 긴 트랙 (참고용)
print("\n=== 가장 긴 트랙 ===")
print(db.run("SELECT Name, Milliseconds/1000.0 as Seconds FROM Track ORDER BY Milliseconds DESC LIMIT 1"))

# 테스트 4: 총 인보이스 금액 (acc_003에서 사용)
print("\n=== 총 인보이스 금액 ===")
print(db.run("SELECT ROUND(SUM(Total), 2) FROM Invoice"))

# 테스트 5 : Employee별 고객수 (acc_004에서 사용)
print("\n=== Employee별 평균 고객 수 ===")
print(db.run("select round(avg(cnt),0) as cnt from (select count(*) as cnt from Customer c join Employee e on c.SupportRepId = e.EmployeeId group by SupportRepId)"))

=== 나라별 고객 수 TOP 5 ===
[('USA', 13), ('Canada', 8), ('France', 5), ('Brazil', 5), ('Germany', 4)]

=== 장르별 트랙 수 TOP 3 ===
[('Rock', 1297), ('Latin', 579), ('Metal', 374)]

=== 가장 긴 트랙 ===
[('Occupation / Precipice', 5286.953)]

=== 총 인보이스 금액 ===
[(2328.6,)]

=== Employee별 평균 고객 수 ===
[(20.0,)]


In [34]:
# Golden Test Set 정의
# 각 테스트 케이스는 하나의 질문-정답 쌍 + 합격 기준으로 구성
golden_tests = [
    # ── Accuracy 테스트: 에이전트가 정확한 숫자/사실을 맞추는가 ──
    {
        "id": "acc_001",
        "category": "accuracy",
        "question": "고객이 가장 많은 나라는 어디인가요?",
        "expected": "USA가 13명으로 가장 많다",
        "expected_keywords": ["USA", "13"],  # 두 키워드 모두 포함해야 통과
        "tolerance": 0.7,
    },
    {
        "id": "acc_002",
        "category": "accuracy",
        "question": "Rock 장르의 트랙은 총 몇 개인가요?",
        "expected": "Rock 장르에는 1297개의 트랙이 있다",
        "expected_keywords": ["1297"],  # 핵심 숫자 하나
        "tolerance": 0.7,
    },
    {
        "id": "acc_003",
        "category": "accuracy",
        "question": "전체 인보이스의 총 금액은 얼마인가요?",
        "expected": "전체 인보이스 총 금액은 2328.60달러이다",
        "expected_keywords": ["2328.60"],
        "tolerance": 0.7,
    },
    {
        "id": "acc_004",
        "category": "accuracy",
        "question": "Employee별 평균 고객 수는 얼마인가요??",
        "expected": "Employee별 평균 고객 수는 약 20명이다",
        "expected_keywords": ["19.666666666666668", "20"],  # 소수점 차이 허용
        "tolerance": 0.7,
    },
    # ── Robustness 테스트: 같은 질문을 다르게 표현해도 같은 답이 나오는가 ──
    # acc_001과 같은 정답이지만, 질문 표현이 다름
    {
        "id": "rob_001",
        "category": "robustness",
        "question": "어느 국가에 고객이 제일 많아?",  # 구어체 변형
        "expected": "USA가 13명으로 가장 많다",
        "expected_keywords": ["USA", "13"],
        "tolerance": 0.7,
    },
    {
        "id": "rob_002",
        "category": "robustness",
        "question": "Which country has the most customers?",  # 영어 변형
        "expected": "USA가 13명으로 가장 많다",
        "expected_keywords": ["USA", "13"],
        "tolerance": 0.7,
    },
    {
        "id": "rob_004",
        "category": "robustness",
        "question": "What is the average number of customers per employee?",  # 영어 변형
        "expected": "The average number of customers per employee is approximately 20",
        "expected_keywords": ["19.666666666666668", "20"],  # 소수점 차이 허용,
        "tolerance": 0.7,
    },
]

print(f"Golden Test Set: {len(golden_tests)}건")
for t in golden_tests:
    print(f"  [{t['id']}] {t['category']:12s} | {t['question'][:40]}...")

Golden Test Set: 7건
  [acc_001] accuracy     | 고객이 가장 많은 나라는 어디인가요?...
  [acc_002] accuracy     | Rock 장르의 트랙은 총 몇 개인가요?...
  [acc_003] accuracy     | 전체 인보이스의 총 금액은 얼마인가요?...
  [acc_004] accuracy     | Employee별 평균 고객 수는 얼마인가요??...
  [rob_001] robustness   | 어느 국가에 고객이 제일 많아?...
  [rob_002] robustness   | Which country has the most customers?...
  [rob_004] robustness   | What is the average number of customers ...


---
## 3. 정량 평가 — Keyword Match & Token F1

가장 기본적인 평가부터 시작합니다. **자동화 가능하고, 비용이 0원**입니다.

| 지표 | 설명 | 장점 | 단점 |
|---|---|---|---|
| **Keyword Match** | 핵심 키워드가 응답에 포함되어 있는가 | 빠르고 명확 | "13명" vs "13" 같은 변형에 취약 |
| **Token F1** | 정답과 응답의 토큰 겹침 비율 | 부분 일치까지 반영 | 의미를 무시 ("13명" ≠ "열세 명") |

### 언제 쓰는가?

- **단답형** 질문 (숫자, 이름 등 정확한 값이 있는 경우)에 적합
- 서술형 응답에는 한계가 있음 → 4장의 LM-as-a-Judge로 보완
- 실무에서는 정량 평가로 **빠르게 걸러내고**, 통과한 것만 LM Judge로 정밀 평가하는 **2단계 파이프라인**을 구성

In [31]:
def keyword_match(response: str, keywords: list[str]) -> float:
    """
    키워드 포함 비율을 반환한다 (0.0 ~ 1.0).
    keywords=["USA", "13"]이고 응답에 둘 다 있으면 1.0, 하나만 있으면 0.5, 둘다 없으면 0.0.
    """
    if not keywords:
        return 1.0
    # 대소문자 무시하고 키워드가 응답 문자열에 포함되는지 확인
    hits = sum(1 for kw in keywords if kw.lower() in response.lower())
    return hits / len(keywords)


def token_f1(reference: str, prediction: str) -> float:
    """
    토큰(단어) 기반 F1 Score.
    reference(정답)와 prediction(에이전트 응답)의 단어 겹침을 측정한다.
    
    F1 = 2 * (precision * recall) / (precision + recall)
    - precision: 에이전트가 말한 단어 중 정답에도 있는 비율
    - recall: 정답 단어 중 에이전트가 말한 비율
    """
    ref_tokens = set(reference.lower().split())
    pred_tokens = set(prediction.lower().split())
    
    common = ref_tokens & pred_tokens  # 교집합
    if not common:
        return 0.0
    
    precision = len(common) / len(pred_tokens)   # 예측 중 맞은 비율
    recall = len(common) / len(ref_tokens)       # 정답 중 맞은 비율
    return 2 * precision * recall / (precision + recall)


# ── 동작 확인 ──
sample_response = "가장 많은 고객이 있는 나라는 USA이며, 고객 수는 13명입니다."
print(f"Keyword Match: {keyword_match(sample_response, ['USA', '13']):.2f}")
# → 1.00 (두 키워드 모두 포함)

print(f"Token F1:      {token_f1('USA가 13명으로 가장 많다', sample_response):.2f}")
# → 0.2~0.3 정도 (겹치는 단어가 일부만 있음 — F1의 한계)

Keyword Match: 1.00
Token F1:      0.15


### 3.1 Golden Test Set에 정량 평가 실행

이제 실제 에이전트를 호출하여 5건의 테스트 케이스를 평가합니다.  
2단계로 진행합니다:
1. **응답 수집** — 에이전트에 질문을 보내고 응답을 저장
2. **채점** — 저장된 응답에 Keyword Match와 F1을 적용

> 에이전트 호출에 시간이 걸립니다 (건당 5~15초). 응답을 `responses` 딕셔너리에 캐싱해두고 이후 여러 평가에서 재사용합니다.

In [35]:
# 에이전트 응답 수집 — 이후 평가에서 계속 재사용할 캐시
# 건당 5~15초 소요, 5건이면 약 1분 정도 걸립니다
responses = {}
for tc in golden_tests:
    print(f"[{tc['id']}] 질문: {tc['question'][:50]}...", end=" ")
    responses[tc["id"]] = ask_agent(tc["question"])
    print("완료")

print(f"\n총 {len(responses)}건 응답 수집 완료")

# ...
# 총 5건 응답 수집 완료

[acc_001] 질문: 고객이 가장 많은 나라는 어디인가요?... 완료
[acc_002] 질문: Rock 장르의 트랙은 총 몇 개인가요?... 완료
[acc_003] 질문: 전체 인보이스의 총 금액은 얼마인가요?... 완료
[acc_004] 질문: Employee별 평균 고객 수는 얼마인가요??... 완료
[rob_001] 질문: 어느 국가에 고객이 제일 많아?... 완료
[rob_002] 질문: Which country has the most customers?... 완료
[rob_004] 질문: What is the average number of customers per employ... 완료

총 7건 응답 수집 완료


In [37]:
# 정량 평가 실행 — Keyword Match와 Token F1 두 지표로 채점
print(f"{'ID':10s} {'카테고리':12s} {'Keyword':>8s} {'F1':>8s} {'통과':>6s}")
print("-" * 50)

quant_results = []
for tc in golden_tests:
    resp = responses[tc["id"]]  # 캐싱된 응답 사용 (에이전트 재호출 없음)
    kw_score = keyword_match(resp, tc["expected_keywords"])
    f1_score = token_f1(tc["expected"], resp)
    passed = kw_score >= tc["tolerance"]  # Keyword Match를 주 판정 기준으로 사용
    
    quant_results.append({
        "id": tc["id"],
        "category": tc["category"],
        "kw_score": kw_score,
        "f1_score": f1_score,
        "passed": passed,
    })
    print(f"{tc['id']:10s} {tc['category']:12s} {kw_score:8.2f} {f1_score:8.2f} {'✅' if passed else '❌':>6s}")

pass_rate = sum(1 for r in quant_results if r["passed"]) / len(quant_results)
print(f"\n통과율: {pass_rate:.0%} ({sum(1 for r in quant_results if r['passed'])}/{len(quant_results)})")

#
# 통과율: 100% (5/5)
#
# 해석: Keyword Match는 높은데 F1은 낮다 — 왜?
# → 에이전트 응답이 정답보다 훨씬 길어서 precision이 낮아지기 때문
# → 이것이 정량 평가의 한계. 의미적으로는 맞지만 F1은 낮게 나옴

ID         카테고리          Keyword       F1     통과
--------------------------------------------------
acc_001    accuracy         1.00     0.12      ✅
acc_002    accuracy         1.00     0.20      ✅
acc_003    accuracy         1.00     0.60      ✅
acc_004    accuracy         0.50     0.13      ❌
rob_001    robustness       1.00     0.12      ✅
rob_002    robustness       1.00     0.11      ✅
rob_004    robustness       0.00     0.00      ❌

통과율: 71% (5/7)


### 🔧 실습 1: 테스트 케이스 추가하기

아래 셀에 **자신만의 테스트 케이스 2개**를 추가하고 평가를 실행하세요.

**진행 순서:**
1. 먼저 `db.run("SELECT ...")` 로 **정답을 확인**
2. 정답에서 핵심 키워드를 뽑기 (숫자, 이름 등)
3. 자연어 질문을 작성
4. `keyword_match`로 평가 실행

**쿼리 예시:**
```sql
SELECT COUNT(*) FROM Employee                    -- 직원 수
SELECT COUNT(DISTINCT GenreId) FROM Genre         -- 장르 수
SELECT Name FROM Album ORDER BY Title LIMIT 3     -- 앨범 이름
SELECT MAX(Total) FROM Invoice                    -- 최대 인보이스 금액
```

In [38]:
# 정답 확인 — 아래 쿼리를 수정하거나 추가하여 테스트 케이스의 정답을 확보하세요
print("직원 수:", db.run("SELECT COUNT(*) FROM Employee"))
# → [(8,)]  ← 직원은 8명

# print("장르 수:", db.run("SELECT COUNT(DISTINCT GenreId) FROM Genre"))
# print("최대 인보이스:", db.run("SELECT MAX(Total) FROM Invoice"))

직원 수: [(8,)]


In [39]:
# TODO: 테스트 케이스 2개를 추가하세요

my_tests = [

    {

        "id": "my_001",

        "category": "accuracy",

        "question": "직원은 총 몇 명인가요?",  # 예시 — 수정하세요

        "expected": "직원은 총 8명이다",

        "expected_keywords": ["8"],

        "tolerance": 0.7,

    },

    # {

    #     "id": "my_002",

    #     "category": "accuracy",

    #     "question": "???",

    #     "expected": "???",

    #     "expected_keywords": ["???"],

    #     "tolerance": 0.7,

    # },

]



for tc in my_tests:

    resp = ask_agent(tc["question"])

    kw = keyword_match(resp, tc["expected_keywords"])

    print(f"[{tc['id']}] Keyword: {kw:.2f} | {'✅' if kw >= tc['tolerance'] else '❌'}")

    print(f"  응답: {resp[:100]}...\n")

[my_001] Keyword: 1.00 | ✅
  응답: 직원은 총 **8명**입니다....



---
## 4. LM-as-a-Judge — 서술형 평가 자동화

Keyword Match는 "USA"와 "13"이 포함되어 있는지만 확인합니다.  
하지만 에이전트가 "미국이 약 13명 정도로 가장 많은 편입니다"라고 대답하면 어떨까요?  
키워드는 맞지만, **"약"**, **"정도"** 같은 표현이 정확한 응답인지는 판단할 수 없습니다.

**LM-as-a-Judge**: 또 다른 LLM에게 "기대 답변과 실제 답변을 비교해서 점수를 매겨줘"라고 요청하는 패턴입니다.

```
┌─────────┐    ┌─────────┐    ┌──────────┐
│Question │───>│  Agent  │───>│ Response │
└─────────┘    └─────────┘    └────┬─────┘
                                   │
                   ┌───────────────┼────────────────┐
                   │           LM Judge             │
                   │  question + expected + actual  │
                   │         → score + reason       │
                   └────────────────────────────────┘
```

### 장점

- **서술형 응답**을 의미 수준에서 평가 가능
- 채점 **이유(reason)**도 함께 반환하므로 디버깅이 쉬움
- 사람 평가 대비 비용이 10~100배 저렴

### 주의

- Judge LLM도 틀릴 수 있음 → 주기적으로 사람이 교차 검증 필요
- `temperature=0`으로 설정하여 평가의 일관성 확보

In [40]:
def lm_judge(
    question: str,
    expected: str,
    actual: str,
    criteria: str = "정확성, 완전성, 명확성"
) -> dict:
    """
    LM-as-a-Judge: 기대 답변과 실제 답변을 비교 채점.
    반환: {"score": 0.0~1.0, "reason": "채점 이유"}
    
    동작 원리:
    1. 프롬프트에 질문, 기대 답변, 실제 답변을 함께 전달
    2. Judge LLM이 두 답변을 비교하여 점수를 매김
    3. JSON으로 파싱하여 score와 reason을 반환
    """
    prompt = f"""당신은 AI 에이전트 응답 품질 평가자입니다.

질문: {question}

기대 답변: {expected}

실제 답변: {actual}

평가 기준: {criteria}
- 핵심 사실이 일치하는가 (숫자, 이름 등)
- 기대 답변의 의미를 충분히 전달하는가
- 잘못된 정보가 포함되어 있지 않은가

0.0(완전히 틀림) ~ 1.0(완벽히 일치) 사이의 점수를 매기세요.
JSON으로만 응답하세요:
{{"score": 0.0, "reason": "이유"}}"""

    response = client.chat.completions.create(
        #model="gpt-4.1-mini", #평가시에는 모델 성능이 좋은 모델은 필요치 않아서...
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},  # JSON 강제
        temperature=0,                             # 일관된 채점을 위해 0
    )
    return json.loads(response.choices[0].message.content)


# ── 테스트 1: 좋은 응답 (의미적으로 일치) ──
good = lm_judge(
    question="고객이 가장 많은 나라는?",
    expected="USA가 13명으로 가장 많다",
    actual="미국(USA)이 13명으로 가장 많은 고객을 보유하고 있습니다.",
)
print(f"좋은 응답 → score: {good['score']}, reason: {good['reason']}")
# ── 테스트 2: 틀린 응답 (핵심 사실 오류) ──
bad = lm_judge(
    question="고객이 가장 많은 나라는?",
    expected="USA가 13명으로 가장 많다",
    actual="브라질이 가장 많은 고객을 보유하고 있습니다.",
)
print(f"틀린 응답 → score: {bad['score']}, reason: {bad['reason']}")

좋은 응답 → score: 1.0, reason: 기대 답변과 실제 답변의 핵심 사실이 완전히 일치합니다. USA가 13명으로 가장 많다는 의미를 정확하고 명확하게 전달하고 있으며, 잘못된 정보도 없습니다.
틀린 응답 → score: 0.0, reason: 기대 답변은 USA가 13명으로 가장 많다고 했지만, 실제 답변은 브라질이라고 하여 국가와 핵심 수치가 모두 불일치합니다. 잘못된 정보가 포함되어 있어 정답 의미를 전달하지 못합니다.


### 4.1 Golden Test Set에 LM Judge 적용

앞서 수집한 `responses`(캐싱된 에이전트 응답)에 LM Judge를 적용합니다.  
**Keyword Match(정량)**와 **LM Judge(정성)** 점수를 나란히 비교해봅니다.

> 두 점수가 일치하면 해당 테스트는 신뢰도가 높습니다.  
> 두 점수가 크게 다르면 (예: KW=1.0인데 Judge=0.5) 키워드는 맞았지만 의미적으로는 문제가 있다는 뜻입니다.

In [41]:
# LM Judge로 전체 테스트 셋 평가 — 이전에 캐싱한 responses를 재사용
print(f"{'ID':10s} {'카테고리':12s} {'KW':>6s} {'Judge':>7s} {'이유'}")
print("-" * 70)

judge_results = []
for tc in golden_tests:
    resp = responses[tc["id"]]  # 에이전트 재호출 없이 캐시 사용
    kw = keyword_match(resp, tc["expected_keywords"])
    judge = lm_judge(tc["question"], tc["expected"], resp)  # LLM API 호출 (건당 ~2초)
    
    judge_results.append({
        "id": tc["id"],
        "category": tc["category"],
        "kw_score": kw,
        "judge_score": judge["score"],
        "reason": judge["reason"],
    })
    print(f"{tc['id']:10s} {tc['category']:12s} {kw:6.2f} {judge['score']:7.2f}  {judge['reason'][:50]}")

avg_judge = sum(r["judge_score"] for r in judge_results) / len(judge_results)
print(f"\nLM Judge 평균: {avg_judge:.2f}")

# ...
#
# LM Judge 평균: 0.90
#
# 주목할 점: KW와 Judge가 모두 높으면 좋은 신호!
# KW는 높은데 Judge가 낮다면 → 키워드는 있지만 문맥이 틀린 경우

ID         카테고리             KW   Judge 이유
----------------------------------------------------------------------
acc_001    accuracy       1.00    1.00  실제 답변이 기대 답변과 동일하게 USA가 13명으로 가장 많다고 정확히 전달하며, 추가 
acc_002    accuracy       1.00    1.00  기대 답변과 실제 답변의 핵심 사실이 완전히 일치합니다. Rock 장르 트랙 수가 1297
acc_003    accuracy       1.00    0.98  핵심 금액 2328.60이 기대 답변과 정확히 일치합니다. 다만 실제 답변에 달러 표기가 
acc_004    accuracy       0.50    0.20  실제 답변은 직원별 고객 수를 나열하고 평균을 계산했지만, 기대 답변의 핵심인 'Emplo
rob_001    robustness     1.00    1.00  실제 답변은 고객이 가장 많은 국가가 USA이며 수가 13명이라고 정확히 제시해 기대 답변
rob_002    robustness     1.00    1.00  기대 답변과 실제 답변이 모두 USA가 고객 수 13명으로 가장 많다고 정확히 일치하며, 
rob_004    robustness     0.00    0.00  기대 답변은 직원 1명당 평균 고객 수가 약 20명이라고 했지만, 실제 답변은 7.375명

LM Judge 평균: 0.74


---
## 5. Faithfulness 측정 — 응답이 근거에 기반하는가

Accuracy는 "정답을 맞췄는가"이고, Faithfulness는 **"근거 있는 말만 했는가"**입니다.

| | Accuracy | Faithfulness |
|---|---|---|
| 비교 대상 | 정답 데이터셋 | 검색/조회된 원본 데이터 |
| 측정 질문 | "맞았는가?" | "근거가 있는가?" |
| 높은 경우 | 정답과 일치 | 출처에만 의존 |
| 낮은 경우 | 정답과 불일치 | 지어낸 정보 포함 (hallucination) |

### 왜 따로 측정하는가?

Accuracy가 높아도 Faithfulness가 낮을 수 있습니다.

**예시**: "USA가 13명으로 가장 많습니다. **최근 3년간 미국 고객이 30% 증가 추세입니다.**"  
→ Accuracy: 높음 (USA, 13명 정답)  
→ Faithfulness: 낮음 ("30% 증가"는 DB에 없는 **지어낸 정보**)

이처럼 정답은 맞추면서도 **없는 사실을 추가하는 hallucination**을 잡으려면 Faithfulness를 별도로 측정해야 합니다. RAG 시스템이나 DB 기반 에이전트에서 특히 중요합니다.

In [42]:
def measure_faithfulness(
    question: str,
    answer: str,
    evidence: str,
) -> dict:
    """
    응답이 evidence(DB 쿼리 결과 등)에만 근거하는지 평가한다.
    RAGAS 스타일의 Faithfulness 측정.
    
    반환:
    - score: 0.0(전혀 근거 없음) ~ 1.0(완전히 근거 있음)
    - grounded_claims: 근거가 확인된 주장 목록
    - ungrounded_claims: 근거 없이 추가된 주장 목록 (hallucination)
    """
    prompt = f"""다음 질문에 대한 답변이 제공된 근거 데이터에만 기반하는지 평가하세요.

질문: {question}

근거 데이터 (DB 쿼리 결과):
{evidence}

답변:
{answer}

평가 기준:
- 답변의 각 주장이 근거 데이터에서 확인 가능한가
- 근거 없이 추가된 정보(hallucination)가 있는가
- 근거 데이터를 왜곡하거나 과장한 부분이 있는가

0.0(전혀 근거 없음) ~ 1.0(완전히 근거 있음)
JSON으로만 응답하세요:
{{"score": 0.0, "grounded_claims": ["..."], "ungrounded_claims": ["..."]}}"""

    response = client.chat.completions.create(
        #model="gpt-4.1-mini",
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)


# ── 테스트 1: 근거 있는 응답 (evidence에 있는 내용만 사용) ──
evidence = "[('USA', 13), ('Canada', 8), ('France', 5), ('Brazil', 5)]"

faithful = measure_faithfulness(
    question="고객이 가장 많은 나라는?",
    answer="USA가 13명으로 가장 많고, 캐나다 8명, 프랑스 5명 순입니다.",
    evidence=evidence,
)
print(f"근거 있는 응답: {faithful['score']}")
print(f"  근거 있는 주장: {faithful.get('grounded_claims', [])}")
# ── 테스트 2: hallucination 포함 응답 (없는 정보 추가) ──
hallucinated = measure_faithfulness(
    question="고객이 가장 많은 나라는?",
    answer="USA가 13명으로 가장 많습니다. 최근 3년간 미국 고객이 30% 증가하는 추세입니다.",
    evidence=evidence,
)
print(f"\nhallucination 포함: {hallucinated['score']}")
print(f"  근거 없는 주장: {hallucinated.get('ungrounded_claims', [])}")

근거 있는 응답: 1.0
  근거 있는 주장: ['USA가 13명으로 가장 많다', '캐나다가 8명이다', '프랑스가 5명이다']

hallucination 포함: 0.5
  근거 없는 주장: ['최근 3년간 미국 고객이 30% 증가하는 추세입니다.']


### 🔧 실습 2: Faithfulness 평가 체험

아래 셀에서 `my_answer`를 수정해서 **점수 변화를 관찰**하세요.

**시도해볼 것:**

| 변형 | 예상 점수 |
|---|---|
| evidence에 있는 내용만 정확히 서술 | 0.9 ~ 1.0 |
| 근거 데이터에 없는 "평균 구매액" 정보를 추가 | 0.5 ~ 0.7 |
| 근거 데이터의 숫자를 살짝 변경 (13 → 15) | 0.3 ~ 0.6 |
| 완전히 근거 없는 내용으로 작성 | 0.0 ~ 0.2 |

In [56]:
evidence = "[('USA', 13), ('Canada', 8), ('France', 5), ('Brazil', 5), ('Korea',6)]"



# TODO: 아래 answer를 수정해보세요

my_answer = "USA가 13명으로 가장 많습니다. 한국은 6명 이라고 판단됩니다."



result = measure_faithfulness(

    question="고객이 가장 많은 나라는?",

    answer=my_answer,

    evidence=evidence,

)

print(f"Faithfulness: {result['score']}")

print(f"근거 있는 주장: {result.get('grounded_claims', [])}")

print(f"근거 없는 주장: {result.get('ungrounded_claims', [])}")

Faithfulness: 1.0
근거 있는 주장: ['USA가 13명으로 가장 많습니다.', '한국은 6명입니다.']
근거 없는 주장: []


---
## 6. Robustness 측정 — 입력 변형에 일관된가

같은 의미의 질문을 **다르게 표현**했을 때 일관된 답을 내는지 측정합니다.

```
"고객이 가장 많은 나라는?"       → USA, 13명  ✅
"어느 국가에 고객이 제일 많아?"   → USA, 13명  ✅ (구어체)
"Which country has most customers?" → USA, 13명  ✅ (영어)
"고객수 기준 상위 국가"           → USA, 13명  ✅ (명사형)
```

4개 변형 **모두** 같은 핵심 정보를 포함해야 Robustness가 높습니다.

### 왜 중요한가?

실제 사용자는 같은 질문을 수십 가지 방식으로 표현합니다.  
개발 중에 테스트한 표현에서만 잘 동작하고, 조금만 바뀌면 실패하는 에이전트는 **프로덕션에서 민원이 쏟아집니다**.

### 측정 방법

같은 정답을 기대하는 N개의 변형 질문을 만들고,  
각각에서 핵심 키워드가 포함되는 비율의 **평균**을 Robustness 점수로 사용합니다.

In [57]:
def measure_robustness(
    variants: list[str],
    expected_keywords: list[str],
    agent_fn=ask_agent,
) -> dict:
    """
    여러 변형 질문에 대해 키워드 일관성을 측정한다.
    모든 변형에서 같은 키워드가 나오면 consistency=1.0.
    
    Args:
        variants: 같은 의미를 가진 질문의 다양한 표현 리스트
        expected_keywords: 모든 변형에서 포함되어야 할 핵심 키워드
        agent_fn: 에이전트 호출 함수
    """
    results = []
    for q in variants:
        resp = agent_fn(q)  # 변형마다 에이전트를 새로 호출
        kw = keyword_match(resp, expected_keywords)
        results.append({"question": q, "response": resp[:100], "kw_score": kw})
    
    # 일관성 = 전체 변형의 키워드 매칭 평균
    consistency = sum(r["kw_score"] for r in results) / len(results)
    return {"consistency": consistency, "details": results}


# ── 테스트: 4가지 표현으로 같은 질문하기 ──
# 에이전트 호출 4번 → 약 30~60초 소요
rob = measure_robustness(
    variants=[
        "고객이 가장 많은 나라는 어디인가요?",     # 문어체
        "어느 국가에 고객이 제일 많아?",           # 구어체
        "Which country has the most customers?",  # 영어
        "고객수 기준 상위 국가를 알려줘",           # 명사형
    ],
    expected_keywords=["USA", "13"],
)

print(f"Robustness (일관성): {rob['consistency']:.2f}")
# → 1.00이면 모든 변형에서 USA와 13을 포함
# → 0.75면 4개 중 3개만 키워드를 포함 (1개 실패)
print()
for d in rob["details"]:
    print(f"  KW={d['kw_score']:.1f} | {d['question'][:30]:30s} → {d['response'][:50]}")

Robustness (일관성): 1.00

  KW=1.0 | 고객이 가장 많은 나라는 어디인가요?           → 고객이 가장 많은 나라는 **미국(USA)** 입니다.  
- **USA: 13명**

참
  KW=1.0 | 어느 국가에 고객이 제일 많아?              → 고객이 가장 많은 국가는 **USA(미국)**입니다.  
- **USA: 13명**
- 그
  KW=1.0 | Which country has the most cus → 가장 많은 고객이 있는 나라는 **USA(미국)** 입니다.  
- **USA: 13명**
  KW=1.0 | 고객수 기준 상위 국가를 알려줘              → 고객 수 기준 상위 국가는 다음과 같습니다. (상위 10개)

1. USA - 13명
2.


### 🔧 실습 3: Robustness 테스트 설계

"Rock 장르의 트랙 수" 질문에 대해 **4가지 이상의 변형**을 만들고 일관성을 측정하세요.  
주석을 해제하고, 원하는 변형을 추가하세요.

**변형 아이디어:**
- 구어체: "록 장르 트랙 수는?"
- 영어: "How many tracks are in the Rock genre?"
- 오타 포함: "Rokc 장르 트랙 몇개?"
- 불필요한 단어: "혹시 Rock이라는 장르에 해당하는 트랙이 도대체 몇 개나 되는 건가요?"
- 대소문자 변형: "rock 장르 트랙 수"

> 오타나 대소문자 변형에서 실패한다면 → 프롬프트에 "오타가 있을 수 있다"는 안내를 추가하는 것을 고려

In [ ]:
# TODO: 변형 질문 4개 이상을 작성하세요

my_variants = [

    "Rock 장르의 트랙은 총 몇 개인가요?",

    # "록 장르 트랙 수는?",

    # "How many tracks are in the Rock genre?",

    # "Rock 장르에 속하는 곡이 몇 개나 되나요?",

]



rob_result = measure_robustness(

    variants=my_variants,

    expected_keywords=["1297"],

)



print(f"Robustness: {rob_result['consistency']:.2f}")

for d in rob_result["details"]:

    print(f"  KW={d['kw_score']:.1f} | {d['question'][:40]}")

---
## 7. Pairwise 비교 — 위치 편향 제거

LM-as-a-Judge에는 치명적인 함정이 있습니다: **먼저 나온 응답을 선호**합니다 (Position Bias).

예를 들어 A와 B 두 응답을 비교할 때:
- [A, B] 순서로 보여주면 → A 선택
- [B, A] 순서로 보여주면 → B 선택 (이번에도 첫 번째를 선택!)

### 해결: 순서를 바꿔 두 번 평가

```
1차 평가: [A, B] 순서 → Judge가 A를 선택
2차 평가: [B, A] 순서 → Judge가 A를 선택 (순서가 바뀌어도 A)
→ 두 번 모두 같은 쪽을 선택했으므로 confidence: high
```

```
1차 평가: [A, B] 순서 → Judge가 A를 선택
2차 평가: [B, A] 순서 → Judge가 B를 선택 (이번엔 B, 즉 첫 번째)
→ 일관성 없음 → confidence: low → tie 처리
```

이 패턴은 실무에서 **프롬프트 A/B 테스트**에 자주 사용됩니다.

In [58]:
def pairwise_judge(
    question: str,
    response_a: str,
    response_b: str,
    criteria: str = "정확성, 간결성, 유용성",
) -> dict:
    """
    두 응답을 비교 평가한다. 위치 편향 제거를 위해 순서를 바꿔 두 번 평가.
    
    반환:
    - winner: "A" | "B" | "tie"
    - confidence: "high" (두 번 일치) | "low" (불일치)
    """
    def single_eval(r1, r2, label1, label2):
        """한 번의 비교 평가 실행"""
        prompt = f"""질문: {question}

응답 {label1}: {r1}

응답 {label2}: {r2}

평가 기준: {criteria}

어느 응답이 더 나은가? JSON으로만 응답하세요:
{{"winner": "{label1}" 또는 "{label2}" 또는 "tie", "reason": "이유"}}"""
        resp = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0,
        )
        return json.loads(resp.choices[0].message.content)

    # 1차 평가: A가 먼저 (원래 순서)
    eval1 = single_eval(response_a, response_b, "A", "B")
    # 2차 평가: B가 먼저 (순서 뒤집기 — 위치 편향 검증)
    eval2 = single_eval(response_b, response_a, "B", "A")

    # 두 번 모두 같은 응답을 선택하면 → 신뢰도 높음
    if eval1["winner"] == "A" and eval2["winner"] == "A":
        return {"winner": "A", "confidence": "high", "eval1": eval1, "eval2": eval2}
    elif eval1["winner"] == "B" and eval2["winner"] == "B":
        return {"winner": "B", "confidence": "high", "eval1": eval1, "eval2": eval2}
    else:
        # 순서에 따라 결과가 달라짐 → 위치 편향 가능성 → tie로 처리
        return {"winner": "tie", "confidence": "low", "eval1": eval1, "eval2": eval2}


# ── 테스트: 간결한 응답 vs 상세한 응답 비교 ──
result = pairwise_judge(
    question="고객이 가장 많은 나라는?",
    response_a="USA가 13명으로 가장 많습니다.",
    response_b="미국(USA)이 13명으로 1위입니다. 2위는 캐나다(8명), 3위는 프랑스(5명)입니다.",
)
print(f"승자: {result['winner']} (신뢰도: {result['confidence']})")
print(f"1차 평가 (A먼저): {result['eval1']['winner']} — {result['eval1']['reason']}")
print(f"2차 평가 (B먼저): {result['eval2']['winner']} — {result['eval2']['reason']}")

# 만약 confidence: low가 나오면 → 위치 편향이 발생한 것

승자: B (신뢰도: high)
1차 평가 (A먼저): B — 응답 B는 정확하게 USA가 13명으로 1위임을 명시하고, 2위와 3위 국가까지 추가 정보를 제공하여 유용성이 높으며, 간결성도 유지하고 있음
2차 평가 (B먼저): B — 응답 B는 정확성과 간결성을 모두 갖추었으며, 1위뿐만 아니라 2위와 3위 국가까지 추가 정보를 제공하여 유용성도 높다.


### 🔧 실습 4: 프롬프트 A/B 테스트

실무에서 자주 하는 질문: **"시스템 프롬프트를 바꾸면 품질이 올라갈까?"**  
Pairwise 비교로 객관적으로 판정합니다.

아래에서 두 에이전트를 만듭니다:
- **Agent A**: 간결한 프롬프트 (최소한의 지시)
- **Agent B**: 상세한 프롬프트 (단계별 지시 + 출력 형식 지정)

같은 질문을 던지고, 어느 프롬프트가 더 나은 응답을 만드는지 비교합니다.

> 결과를 보고 나서, 프롬프트를 직접 수정하여 더 나은 Agent C를 만들어보세요!

In [59]:
# ── 에이전트 A: 간결한 프롬프트 (최소 지시) ──
agent_a = create_agent(
    model=llm, tools=tools,
    system_prompt=f"SQL 에이전트. 한국어로 간결하게 답변. LIMIT 10. Dialect: {db.dialect}",
)

# ── 에이전트 B: 상세한 프롬프트 (단계별 지시 + 형식 지정) ──
agent_b = create_agent(
    model=llm, tools=tools,
    system_prompt=(
        "당신은 데이터 분석 전문가입니다. SQL을 사용하여 질문에 답변합니다.\n"
        "반드시 1) 테이블 확인 2) 스키마 확인 3) 쿼리 검증 4) 실행 순서를 따르세요.\n"
        "결과는 표 형식으로 정리하고, 핵심 인사이트를 덧붙이세요.\n"
        f"LIMIT 10 필수. DML 금지. Dialect: {db.dialect}"
    ),
)

# 같은 질문으로 두 에이전트의 응답을 비교
question = "매출이 가장 높은 아티스트 3명은?"
resp_a = agent_a.invoke({"messages": [{"role": "user", "content": question}]})["messages"][-1].content
resp_b = agent_b.invoke({"messages": [{"role": "user", "content": question}]})["messages"][-1].content

print("=== Agent A (간결한 프롬프트) ===")
print(resp_a[:300])
print("\n=== Agent B (상세한 프롬프트) ===")
print(resp_b[:300])

=== Agent A (간결한 프롬프트) ===
매출이 가장 높은 아티스트 3명은 다음과 같습니다.

1. **Iron Maiden** — **138.60**
2. **U2** — **105.93**
3. **Metallica** — **90.09**

원하시면 전체 순위도 보여드릴게요.

=== Agent B (상세한 프롬프트) ===
테이블을 확인했고, 스키마를 검증한 뒤 쿼리를 실행했습니다.  
매출이 가장 높은 아티스트 3명은 아래와 같습니다.

| 순위 | 아티스트 | 매출 |
|---|---|---:|
| 1 | Iron Maiden | 138.60 |
| 2 | U2 | 105.93 |
| 3 | Metallica | 90.09 |

핵심 인사이트:
- **Iron Maiden**이 1위로, 2위 **U2**보다 약 **32.67** 더 높은 매출을 기록했습니다.
- 상위 3팀 모두 록/메탈 계열 아티스트로, 이 데이터셋에서 해당 장르의 구매 비중이 높


In [60]:
# Pairwise 비교 — 순서를 바꿔 두 번 평가
pw = pairwise_judge(question, resp_a, resp_b)
print(f"승자: {pw['winner']} (신뢰도: {pw['confidence']})")
print(f"이유: {pw['eval1']['reason']}")

# 해석:
# - winner: "A"면 간결한 프롬프트가 우세, "B"면 상세한 프롬프트가 우세
# - confidence: "high"면 순서를 바꿔도 동일한 결과 → 신뢰할 수 있음
# - confidence: "low"면 위치 편향 가능성 → 추가 검증 필요

승자: B (신뢰도: high)
이유: 응답 B는 매출 상위 3명의 아티스트를 표로 명확히 제시하고, 핵심 인사이트를 추가하여 데이터 해석에 도움을 주며, 쿼리 방식과 데이터 출처를 설명해 신뢰성을 높임. 또한 추가 요청 옵션을 제안해 유용성도 높음. 반면 응답 A는 간결하지만 정보가 부족하고 부가 설명이 없어 정확성과 유용성 면에서 떨어짐.


---
## 8. 복합 대시보드 — 배포 판단 기준

하나의 지표만 보면 다른 영역이 악화됩니다.

**실제 사례**: Accuracy만 올리기 위해 프롬프트를 과도하게 구체적으로 만들었더니,  
→ 조금만 다른 표현에 실패 (Robustness 하락)  
→ 길고 상세한 답변에 hallucination 증가 (Faithfulness 하락)

**반드시 복합 지표 대시보드로 관리합니다.**

```
릴리즈 기준 예시:
  Accuracy     ≥ 0.80  → 정답률
  Faithfulness ≥ 0.75  → 근거 기반 응답 비율
  Robustness   ≥ 0.85  → 표현 변형 일관성
  → 3개 모두 통과해야 배포 가능
```

아래에서 Golden Test Set 전체를 한 번에 평가하고, 카테고리별 점수를 대시보드로 출력합니다.

In [61]:
def run_full_evaluation(test_cases, agent_fn=ask_agent):
    """
    Golden Test Set 전체를 평가하는 파이프라인.
    각 테스트 케이스에 대해:
    1. 에이전트 응답 수집
    2. 정량 평가 (Keyword Match, Token F1)
    3. LM Judge 채점
    결과를 리스트로 반환한다.
    """
    results = []
    for tc in test_cases:
        print(f"  평가 중: [{tc['id']}] {tc['question'][:40]}...", end="")
        
        # Step 1: 에이전트 응답 수집
        resp = agent_fn(tc["question"])
        
        # Step 2: 정량 평가
        kw = keyword_match(resp, tc["expected_keywords"])
        f1 = token_f1(tc["expected"], resp)
        
        # Step 3: LM Judge 채점
        judge = lm_judge(tc["question"], tc["expected"], resp)
        
        results.append({
            "id": tc["id"],
            "category": tc["category"],
            "kw_score": kw,
            "f1_score": f1,
            "judge_score": judge["score"],
            "response": resp[:200],
        })
        print(f" KW={kw:.1f} Judge={judge['score']:.2f}")
    
    return results


# 전체 Golden Test Set(5건) 평가 실행 — 약 2~3분 소요
print("=== 전체 평가 실행 ===")
all_results = run_full_evaluation(golden_tests)

=== 전체 평가 실행 ===
  평가 중: [acc_001] 고객이 가장 많은 나라는 어디인가요?... KW=1.0 Judge=1.00
  평가 중: [acc_002] Rock 장르의 트랙은 총 몇 개인가요?... KW=1.0 Judge=1.00
  평가 중: [acc_003] 전체 인보이스의 총 금액은 얼마인가요?... KW=1.0 Judge=0.98
  평가 중: [acc_004] Employee별 평균 고객 수는 얼마인가요??... KW=0.5 Judge=0.20
  평가 중: [rob_001] 어느 국가에 고객이 제일 많아?... KW=1.0 Judge=1.00
  평가 중: [rob_002] Which country has the most customers?... KW=1.0 Judge=1.00
  평가 중: [rob_004] What is the average number of customers ... KW=0.0 Judge=0.00


In [62]:
def print_dashboard(results, thresholds=None):
    """
    카테고리별 평균 점수를 계산하고, threshold 기준으로 PASS/FAIL 판정.
    모든 카테고리가 통과해야 RELEASE READY.
    """
    if thresholds is None:
        thresholds = {"accuracy": 0.80, "robustness": 0.85}
    
    # 카테고리별 집계
    categories = {}
    for r in results:
        cat = r["category"]
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(r)
    
    print("\n" + "=" * 55)
    print("          AGENT EVALUATION DASHBOARD")
    print("=" * 55)
    
    all_pass = True
    for cat, items in categories.items():
        avg_kw = sum(r["kw_score"] for r in items) / len(items)
        avg_judge = sum(r["judge_score"] for r in items) / len(items)
        threshold = thresholds.get(cat, 0.75)
        passed = avg_judge >= threshold
        if not passed:
            all_pass = False
        
        status = "PASS" if passed else "FAIL"
        print(f"\n  [{cat.upper():12s}]  (기준: {threshold:.0%})")
        print(f"    Keyword Match : {avg_kw:.2f}")
        print(f"    LM Judge      : {avg_judge:.2f}  → {status}")
        print(f"    테스트 수     : {len(items)}건")
    
    print("\n" + "-" * 55)
    overall = "RELEASE READY" if all_pass else "NOT READY"
    print(f"  배포 판단: {overall}")
    print("=" * 55)


# 대시보드 출력
print_dashboard(all_results)

# =======================================================
#           AGENT EVALUATION DASHBOARD
# =======================================================
#
#   [ACCURACY    ]  (기준: 80%)
#     Keyword Match : 1.00
#     LM Judge      : 0.92  → PASS
#     테스트 수     : 3건
#
#   [ROBUSTNESS  ]  (기준: 85%)
#     Keyword Match : 1.00
#     LM Judge      : 0.90  → PASS
#     테스트 수     : 2건
#
# -------------------------------------------------------
#   배포 판단: RELEASE READY
# =======================================================


          AGENT EVALUATION DASHBOARD

  [ACCURACY    ]  (기준: 80%)
    Keyword Match : 0.88
    LM Judge      : 0.80  → FAIL
    테스트 수     : 4건

  [ROBUSTNESS  ]  (기준: 85%)
    Keyword Match : 0.67
    LM Judge      : 0.67  → FAIL
    테스트 수     : 3건

-------------------------------------------------------
  배포 판단: NOT READY


### 🔧 실습 5: 나만의 평가 파이프라인

**종합 실습**입니다. 지금까지 배운 모든 것을 조합합니다.

**진행 순서:**

1. `db.run()`으로 **정답 확인** (아래 셀)
2. Golden Test Set을 **10건으로 확장** (accuracy 3건 + robustness 2건 추가)
3. `run_full_evaluation()`으로 **평가 실행**
4. `print_dashboard()`로 **배포 판단** (threshold를 직접 설정)

**테스트 케이스 아이디어:**

| 질문 | 확인용 SQL |
|---|---|
| 앨범이 가장 많은 아티스트는? | `SELECT ar.Name, COUNT(*) FROM Album al JOIN Artist ar ON al.ArtistId=ar.ArtistId GROUP BY ar.Name ORDER BY COUNT(*) DESC LIMIT 3` |
| 가장 비싼 트랙은? | `SELECT Name, UnitPrice FROM Track ORDER BY UnitPrice DESC LIMIT 1` |
| 2009년 인보이스 총액은? | `SELECT SUM(Total) FROM Invoice WHERE InvoiceDate LIKE '2009%'` |

In [ ]:
# 정답 확인 — 테스트 케이스를 만들기 전에 DB에서 직접 확인하세요
print("아티스트 수:", db.run("SELECT COUNT(DISTINCT ArtistId) FROM Artist"))
print("아티스트 예시:", db.run("SELECT Name FROM Artist LIMIT 5"))

# 아래 주석을 해제하고 필요한 쿼리를 실행하세요
# print("앨범 많은 아티스트:", db.run("SELECT ar.Name, COUNT(*) as cnt FROM Album al JOIN Artist ar ON al.ArtistId=ar.ArtistId GROUP BY ar.Name ORDER BY cnt DESC LIMIT 3"))
# print("비싼 트랙:", db.run("SELECT Name, UnitPrice FROM Track ORDER BY UnitPrice DESC LIMIT 1"))
# print("2009년 매출:", db.run("SELECT ROUND(SUM(Total),2) FROM Invoice WHERE InvoiceDate LIKE '2009%'"))

In [ ]:
# TODO: 기존 5건 + 추가 5건 = 총 10건으로 확장하세요
extended_tests = golden_tests + [
    # accuracy 테스트 3건 추가
    # {
    #     "id": "acc_004",
    #     "category": "accuracy",
    #     "question": "앨범이 가장 많은 아티스트는 누구인가요?",
    #     "expected": "Iron Maiden이 21개로 가장 많다",
    #     "expected_keywords": ["Iron Maiden"],
    #     "tolerance": 0.7,
    # },
    # {
    #     "id": "acc_005",
    #     "category": "accuracy",
    #     "question": "???",
    #     "expected": "???",
    #     "expected_keywords": ["???"],
    #     "tolerance": 0.7,
    # },
    # {
    #     "id": "acc_006",
    #     "category": "accuracy",
    #     "question": "???",
    #     "expected": "???",
    #     "expected_keywords": ["???"],
    #     "tolerance": 0.7,
    # },
    # robustness 테스트 2건 추가
    # {
    #     "id": "rob_003",
    #     "category": "robustness",
    #     "question": "록 음악 트랙이 몇 개야?",  # acc_002의 구어체 변형
    #     "expected": "Rock 장르에는 1297개의 트랙이 있다",
    #     "expected_keywords": ["1297"],
    #     "tolerance": 0.7,
    # },
    # {
    #     "id": "rob_004",
    #     "category": "robustness",
    #     "question": "How many Rock tracks?",  # 영어 변형
    #     "expected": "Rock 장르에는 1297개의 트랙이 있다",
    #     "expected_keywords": ["1297"],
    #     "tolerance": 0.7,
    # },
]

# 주석 해제 후 실행하세요
# ext_results = run_full_evaluation(extended_tests)
# print_dashboard(ext_results, thresholds={"accuracy": 0.80, "robustness": 0.85})

---
## 9. 주의사항 정리

### LM-as-a-Judge 편향 3가지

| 편향 | 설명 | 이 노트북에서의 대응 |
|---|---|---|
| **위치 편향** | 먼저 나온 응답을 선호 | `pairwise_judge()`에서 순서를 바꿔 두 번 평가 |
| **자기 선호** | 같은 모델이 생성한 응답을 선호 | Agent(gpt-4.1-mini)와 Judge(gpt-4.1-mini)가 같음 — 실무에서는 분리 권장 |
| **보상 해킹** | 평가 기준에 과적합된 응답 | 기준을 주기적으로 변경 + 사람 교차 검증 |

### Golden Test Set 관리 원칙

- 학습 데이터에 포함되면 측정값이 **오염**된다 → 격리된 저장소에서 버전 관리
- 6개월마다 새 예제를 추가하고 오래된 예제를 갱신
- 카테고리별로 균형 있게 구성 (시작 시 20~30건, 목표 200~500건)
- 정답이 바뀔 수 있는 데이터(실시간 API 등)는 주기적으로 정답 갱신

### 실무 적용 체크리스트

- [ ] 단답형은 EM/F1로 빠르게 거르기 (비용 0원, 밀리초)
- [ ] 서술형은 LM-as-a-Judge로 자동 평가 (건당 ~$0.01, ~2초)
- [ ] 분기마다 사람이 50건 이상 샘플링하여 Judge와 교차 검증
- [ ] 하나의 지표만 보지 않기 — 복합 대시보드 필수
- [ ] 운영 중 전수 평가 대신 5~10% 샘플링으로 비용 관리
- [ ] CI/CD에 Golden Test 통합 → 코드 변경 시 자동 회귀 테스트

---
## 핵심 정리

| 구성 요소 | 이 노트북에서 구현한 것 | 실무 사용 시점 |
|---|---|---|
| **Golden Test Set** | Chinook DB 기반 5건 (accuracy 3 + robustness 2) | 개발 초기부터 누적 |
| **정량 평가** | `keyword_match()`, `token_f1()` | 매 커밋, CI/CD |
| **LM-as-a-Judge** | `lm_judge()` — 기대 vs 실제 응답 채점 | 매 릴리즈 전 |
| **Faithfulness** | `measure_faithfulness()` — 근거 기반 평가 | RAG/DB 에이전트 운영 중 |
| **Robustness** | `measure_robustness()` — 변형 질문 일관성 | 프롬프트 변경 후 |
| **Pairwise 비교** | `pairwise_judge()` — 위치 편향 제거 | 프롬프트 A/B 테스트 |
| **대시보드** | `print_dashboard()` — 배포 판단 기준 | 릴리즈 게이트 |

### 평가 파이프라인 흐름

```
Golden Test Set
    │
    ▼
Agent 응답 수집 ──────────────────────────────────────────┐
    │                                                      │
    ├─→ 정량 평가 (Keyword Match / F1)  ← 빠르고 무료      │
    │     └─ 실패 시: 즉시 FAIL                            │
    │                                                      │
    ├─→ LM Judge (서술형 채점)  ← 건당 ~$0.01              │
    │     └─ score + reason 반환                           │
    │                                                      │
    ├─→ Faithfulness (근거 확인)  ← hallucination 탐지     │
    │     └─ grounded / ungrounded claims                  │
    │                                                      │
    ├─→ Robustness (변형 일관성)  ← 여러 표현으로 테스트    │
    │     └─ consistency score                             │
    │                                                      │
    ▼                                                      │
Dashboard ──→ PASS / FAIL ──→ 배포 판단                     │
                                                           │
Pairwise 비교 (A/B 테스트) ← 프롬프트 개선 시 사용 ────────┘
```